# Chapter 13: The Policy Engine — Runtime Rule Enforcement

**Learning objectives:**
- Define `PolicyRule` objects with action patterns, requirements and severity levels
- Use `PolicyEngine` to validate tool calls through a four-layer GMS-backed check
- Integrate the policy engine with `ToolGateway` for complete pre-execution checking
- Inspect `PolicyDecision` results including blocking rules and geometric scores

---
## Preamble

Chapter 12 introduced the `ToolGateway` — a control layer that intercepts tool calls to inject faults, mock responses and stale data. The gateway makes *infrastructure*-level decisions: "should this tool succeed or fail?" But it does not ask the *governance* question: "is this tool call *permitted* given the current policy?"

The **PolicyEngine** answers that question. It validates every proposed tool call against a set of `PolicyRule` objects through four layers of verification:

| Layer | Mechanism | Purpose |
|---|---|---|
| 1. Explicit rules | `fnmatch` pattern matching | Hard-coded requires/blocks/allows |
| 2. GMS graph lookup | `query_triples()` | Discover rules encoded in the knowledge graph |
| 3. Geometric plausibility | `score_triple()` | Is this action plausible in the given context? |
| 4. Contradiction detection | `tension_energy()` | Does this action contradict any blocking rule? |

Each check returns a `PolicyDecision` with pass/fail, blocking rule IDs, geometric scores and a human-readable reason. When integrated with the gateway, policy violations are blocked *before* any fault injection or mock substitution takes place.

In [ ]:
# === Preamble (all notebooks) ===
import os
import warnings
warnings.filterwarnings("ignore")

import torch
import numpy as np
from collections import Counter

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

In [ ]:
# [AUTO-INJECTED] Disable DOE tracing for parallel execution safety
from knowlytix.harness.doe import DOEHarnessConfig
DOEHarnessConfig.__dataclass_fields__['enable_tracing'].default = False


In [ ]:
from importlib.resources import files
# Load the Chapter 4 store
from knowlytix.harness.doe import DOEGMSBenchmark, DOEHarnessConfig

config = DOEHarnessConfig(
    markdown_path=str(files("knowlytix.harness.doe").joinpath("tests/model_risk_assessment.md")),
    stores_dir="../lab_stores",
    store_name="ch04_ground_truth",
    n_runs=4,
    ingest_epochs=200,
    device=str(DEVICE),
)
bench = DOEGMSBenchmark(config)
store = bench.ingest()
print(f"Store: {store.stats()['entities']} entities, "
      f"{store.stats()['triples']} triples, "
      f"{store.stats()['enm_entries']} ENM")

---

## PolicyRule: Defining Governance Constraints

A `PolicyRule` encodes a single governance constraint. Rules are matched against tool calls by `fnmatch` glob pattern — the same pattern language used by `FaultProfile` in Chapter 12.

**PolicyRule fields:**

| Field | Type | Default | Purpose |
|---|---|---|---|
| `rule_id` | str | `""` | Unique identifier (e.g. `"R-001"`) |
| `action` | str | `""` | Tool name or fnmatch glob (e.g. `"delete_*"`) |
| `requirement` | str | `""` | What is required or blocked |
| `rule_type` | str | `"requires"` | `requires`, `blocks`, or `allows` |
| `context` | str | `"*"` | Scenario context pattern (fnmatch) |
| `severity` | str | `"error"` | `error` (hard block) or `warning` (log only) |

**Rule types:**

| Type | Meaning | Example |
|---|---|---|
| `requires` | Action needs this requirement to proceed | `submit_escalation` requires `human_approval` |
| `blocks` | Action is blocked when requirement matches | `dangerous_delete` blocked in all contexts |
| `allows` | Explicitly permits the action (overrides plausibility) | `data_lookup` allowed for `authenticated` users |

In [ ]:
# Create three policy rules demonstrating each rule type
from knowlytix.harness.doe.policy import PolicyRule, PolicyDecision

# 1. Requires: escalation needs human approval
r1 = PolicyRule(
    rule_id="R-001",
    action="submit_escalation",
    requirement="human_approval",
    rule_type="requires",
    severity="error",
)

# 2. Blocks: dangerous_delete blocked everywhere
r2 = PolicyRule(
    rule_id="R-002",
    action="dangerous_delete",
    requirement="*",
    rule_type="blocks",
    severity="error",
)

# 3. Allows: data_lookup explicitly permitted
r3 = PolicyRule(
    rule_id="R-003",
    action="data_lookup",
    requirement="authenticated",
    rule_type="allows",
    severity="warning",
)

rules = [r1, r2, r3]
for r in rules:
    print(f"{r.rule_id}  action={r.action:25s}  "
          f"type={r.rule_type:8s}  severity={r.severity}")

In [5]:
# YAML serialization round-trip
import yaml

yaml_data = {"rules": [r.to_dict() for r in rules]}
yaml_str = yaml.dump(yaml_data, default_flow_style=False, sort_keys=False)
print("=== Policy Rules YAML ===")
print(yaml_str)

# Round-trip: deserialize back
loaded = yaml.safe_load(yaml_str)
recon = [PolicyRule.from_dict(d) for d in loaded["rules"]]
print(f"Round-trip: {len(recon)} rules recovered")
for r in recon:
    print(f"  {r.rule_id}: {r.action} {r.rule_type} {r.requirement}")

=== Policy Rules YAML ===
rules:
- rule_id: R-001
  action: submit_escalation
  requirement: human_approval
  rule_type: requires
  context: '*'
  severity: error
- rule_id: R-002
  action: dangerous_delete
  requirement: '*'
  rule_type: blocks
  context: '*'
  severity: error
- rule_id: R-003
  action: data_lookup
  requirement: authenticated
  rule_type: allows
  context: '*'
  severity: warning

Round-trip: 3 rules recovered
  R-001: submit_escalation requires human_approval
  R-002: dangerous_delete blocks *
  R-003: data_lookup allows authenticated


---

## PolicyDecision: Structured Check Results

Every call to `PolicyEngine.check()` returns a `PolicyDecision` that captures the full audit trail.

**PolicyDecision fields:**

| Field | Type | Purpose |
|---|---|---|
| `allowed` | bool | Whether the action is permitted |
| `action` | str | The action that was checked |
| `rules_checked` | list[str] | IDs of all rules evaluated |
| `blocking_rules` | list[str] | IDs of rules that blocked the action |
| `gms_scores` | dict[str, float] | Geometric scores (plausibility, tension) |
| `reason` | str | Human-readable explanation |

---

## The Four-Layer Check Pipeline

When `PolicyEngine.check(action, args, context)` is called, it runs four layers of verification in sequence:

1. **Explicit rule matching** — All `PolicyRule` objects whose `action` and `context` patterns match (via `fnmatch`) are collected. `requires` and `blocks` rules with `severity="error"` add to the blocking list.

2. **GMS graph lookup** — The engine calls `store.query_triples(head=action_entity)` to discover any additional constraints encoded in the knowledge graph. Triples with relation `blocks` or `requires` are treated as implicit rules.

3. **Geometric plausibility** — If a context is provided, the engine scores `(action, "allowed_in", context)` via `store.score_triple()`. A geodesic distance above `PLAUSIBILITY_THRESHOLD` (1.5) means the action is implausible in that context and is blocked.

4. **Contradiction detection** — For each blocking rule entity, the engine measures `store.tension_energy(action, rule)`. Tension above `TENSION_THRESHOLD` (1.7) indicates a strong contradiction — the action fundamentally conflicts with the rule.

The final decision is `allowed = (len(blocking_rules) == 0)`.

---

## Lab 13.1 — Define and Enforce Policies

In this lab we create a `PolicyEngine` with three rules and check a variety of actions to observe how each rule type behaves.

In [ ]:
# Lab 13.1a — Create a PolicyEngine with three rules
from knowlytix.harness.doe.policy import PolicyEngine

rules = [
    PolicyRule(
        rule_id="R-001", action="submit_escalation",
        requirement="human_approval", rule_type="requires",
        severity="error",
    ),
    PolicyRule(
        rule_id="R-002", action="dangerous_delete",
        requirement="*", rule_type="blocks",
        severity="error",
    ),
    PolicyRule(
        rule_id="R-003", action="data_lookup",
        requirement="authenticated", rule_type="allows",
        severity="warning",
    ),
]

# Helper: create PolicyEngine without GPU learning.
# The stored GMS has a fixed embedding matrix; adding new entities via
# RuntimeLearner would exceed its capacity.  Disabling the learner
# keeps explicit rule matching (Layer 1) and GMS graph lookup (Layer 2)
# fully functional while skipping the geometric encoding step.
def make_policy_engine(s, r):
    eng = PolicyEngine(store=s, rules=None)
    eng._learner = None
    eng.encode_rules(r)
    return eng

engine = make_policy_engine(store, rules)
print(f"PolicyEngine created with {len(engine._rules)} rules")
print(f"Thresholds: plausibility={engine.PLAUSIBILITY_THRESHOLD}, "
      f"tension={engine.TENSION_THRESHOLD}")

In [7]:
# Lab 13.1b — Check each action

# 1. submit_escalation — should be BLOCKED (requires human_approval)
d1 = engine.check("submit_escalation", {"finding": "degraded model"})
print("=== Check: submit_escalation ===")
print(f"  allowed        : {d1.allowed}")
print(f"  rules_checked  : {d1.rules_checked}")
print(f"  blocking_rules : {d1.blocking_rules}")
print(f"  reason         : {d1.reason}")
if d1.gms_scores:
    print(f"  gms_scores     : {d1.gms_scores}")

=== Check: submit_escalation ===
  allowed        : False
  rules_checked  : ['R-001']
  blocking_rules : ['R-001']
  reason         : Rule R-001: 'submit_escalation' requires 'human_approval' (not met)


In [8]:
# 2. dangerous_delete — should be BLOCKED (blocks rule)
d2 = engine.check("dangerous_delete", {"target": "all"})
print("=== Check: dangerous_delete ===")
print(f"  allowed        : {d2.allowed}")
print(f"  rules_checked  : {d2.rules_checked}")
print(f"  blocking_rules : {d2.blocking_rules}")
print(f"  reason         : {d2.reason}")
if d2.gms_scores:
    print(f"  gms_scores     : {d2.gms_scores}")

=== Check: dangerous_delete ===
  allowed        : False
  rules_checked  : ['R-002']
  blocking_rules : ['R-002']
  reason         : Rule R-002: 'dangerous_delete' blocked by '*' in context '*'


In [9]:
# 3. data_lookup — should be ALLOWED (explicit allows rule)
d3 = engine.check("data_lookup", {"entity": "MDL-001"})
print("=== Check: data_lookup ===")
print(f"  allowed        : {d3.allowed}")
print(f"  rules_checked  : {d3.rules_checked}")
print(f"  blocking_rules : {d3.blocking_rules}")
print(f"  reason         : {d3.reason}")
if d3.gms_scores:
    print(f"  gms_scores     : {d3.gms_scores}")

=== Check: data_lookup ===
  allowed        : True
  rules_checked  : ['R-003']
  blocking_rules : []
  reason         : Explicitly allowed by policy rule


In [10]:
# 4. risk_check — no explicit rule; falls through to GMS layers
d4 = engine.check("risk_check", {"model": "MDL-001"})
print("=== Check: risk_check (no explicit rule) ===")
print(f"  allowed        : {d4.allowed}")
print(f"  rules_checked  : {d4.rules_checked}")
print(f"  blocking_rules : {d4.blocking_rules}")
print(f"  reason         : {d4.reason}")
if d4.gms_scores:
    print(f"  gms_scores     : {d4.gms_scores}")

=== Check: risk_check (no explicit rule) ===
  allowed        : True
  rules_checked  : []
  blocking_rules : []
  reason         : Action permitted: no blocking rules matched


In [11]:
# 5. Summary table of all four checks
print(f"{'Action':<25s} {'Allowed':<10s} {'Rules Checked':<18s} {'Blocking':<20s}")
print("-" * 75)
for d in [d1, d2, d3, d4]:
    rc = ", ".join(d.rules_checked) if d.rules_checked else "(none)"
    br = ", ".join(d.blocking_rules) if d.blocking_rules else "(none)"
    print(f"{d.action:<25s} {str(d.allowed):<10s} {rc:<18s} {br:<20s}")

Action                    Allowed    Rules Checked      Blocking            
---------------------------------------------------------------------------
submit_escalation         False      R-001              R-001               
dangerous_delete          False      R-002              R-002               
data_lookup               True       R-003              (none)              
risk_check                True       (none)             (none)              


### Context-Sensitive Rules

Rules can be scoped to specific contexts using the `context` field. A rule with `context="production"` only matches when the caller passes `context="production"` to `check()`.

In [12]:
# Context-sensitive rule: block delete_* only in production
ctx_rules = [
    PolicyRule(
        rule_id="CTX-001", action="delete_*",
        requirement="admin_context", rule_type="blocks",
        context="production", severity="error",
    ),
    PolicyRule(
        rule_id="CTX-002", action="delete_*",
        requirement="cleanup", rule_type="allows",
        context="staging", severity="warning",
    ),
]

ctx_engine = make_policy_engine(store, ctx_rules)

# delete_records in production — should be BLOCKED
d_prod = ctx_engine.check("delete_records", {}, context="production")
print(f"delete_records in production: allowed={d_prod.allowed}, "
      f"blocking={d_prod.blocking_rules}")

# delete_records in staging — should be ALLOWED
d_stag = ctx_engine.check("delete_records", {}, context="staging")
print(f"delete_records in staging:    allowed={d_stag.allowed}, "
      f"blocking={d_stag.blocking_rules}")

# delete_records with no context — neither rule matches
d_none = ctx_engine.check("delete_records", {})
print(f"delete_records (no context):  allowed={d_none.allowed}, "
      f"blocking={d_none.blocking_rules}")

delete_records in production: allowed=False, blocking=['CTX-001']
delete_records in staging:    allowed=True, blocking=[]
delete_records (no context):  allowed=True, blocking=[]


### YAML Loading

Rules can be loaded from YAML files for repeatable, version-controlled policy definitions.

In [ ]:
# Write rules to a temporary YAML file, then load via load_policy_rules()
import tempfile
from knowlytix.harness.doe.policy import load_policy_rules

yaml_content = yaml.dump(
    {"rules": [r.to_dict() for r in rules]},
    default_flow_style=False, sort_keys=False,
)

with tempfile.NamedTemporaryFile(mode="w", suffix=".yaml", delete=False) as f:
    f.write(yaml_content)
    yaml_path = f.name

loaded_rules = load_policy_rules(yaml_path)
print(f"Loaded {len(loaded_rules)} rules from YAML:")
for r in loaded_rules:
    print(f"  {r.rule_id}: {r.action} {r.rule_type} {r.requirement} "
          f"(severity={r.severity})")

os.unlink(yaml_path)

---

## GMS Encoding: Rules as Geometric Triples

When rules are loaded into a `PolicyEngine`, the engine calls `encode_rules()` to convert each rule into four GMS triples via the `RuntimeLearner`:

| Triple | Example |
|---|---|
| `(action, rule_type, requirement)` | `(submit_escalation, requires, human_approval)` |
| `(rule_id, is_type, policy_rule)` | `(R-001, is_type, policy_rule)` |
| `(rule_id, applies_to, action)` | `(R-001, applies_to, submit_escalation)` |
| `(rule_id, severity, level)` | `(R-001, severity, error)` |

This encoding enables the GMS graph lookup (Layer 2) and geometric scoring (Layers 3–4) — the same primitives used for knowledge verification in Chapters 2–6 now enforce runtime policy.

In [14]:
# Demonstrate the encode_rules triple structure
# (GPU learning is disabled, but we can show what triples *would* be created)

print("=== Triples generated by encode_rules() ===")
print(f"{'Rule':<8s} {'Triple':<60s}")
print("-" * 70)
for r in rules:
    triples = [
        (r.action, r.rule_type, r.requirement),
        (r.rule_id, "is_type", "policy_rule"),
        (r.rule_id, "applies_to", r.action),
        (r.rule_id, "severity", r.severity),
    ]
    for h, rel, t in triples:
        print(f"{r.rule_id:<8s} ({h}, {rel}, {t})")
print(f"\nTotal: {len(rules) * 4} triples for {len(rules)} rules")

=== Triples generated by encode_rules() ===
Rule     Triple                                                      
----------------------------------------------------------------------
R-001    (submit_escalation, requires, human_approval)
R-001    (R-001, is_type, policy_rule)
R-001    (R-001, applies_to, submit_escalation)
R-001    (R-001, severity, error)
R-002    (dangerous_delete, blocks, *)
R-002    (R-002, is_type, policy_rule)
R-002    (R-002, applies_to, dangerous_delete)
R-002    (R-002, severity, error)
R-003    (data_lookup, allows, authenticated)
R-003    (R-003, is_type, policy_rule)
R-003    (R-003, applies_to, data_lookup)
R-003    (R-003, severity, warning)

Total: 12 triples for 3 rules


---

## Lab 13.2 — Gateway + Policy Integration

The `ToolGateway` accepts an optional `policy_engine` parameter. When provided, the gateway calls `policy_engine.check()` as the *first* step in its `_before_tool` hook — before any fault injection or mock substitution.

The gateway expects `check()` to return a boolean, but `PolicyEngine.check()` returns a `PolicyDecision` object. We bridge the two with a thin adapter that extracts the `allowed` flag and keeps the full decision for inspection.

In [15]:
# PolicyAdapter: bridge between PolicyEngine and ToolGateway
class PolicyAdapter:
    """Adapts PolicyEngine for ToolGateway (which expects check() -> bool)."""

    def __init__(self, engine: PolicyEngine):
        self.engine = engine
        self.decisions: list[PolicyDecision] = []

    def check(self, action: str, args: dict, context: str = "") -> bool:
        decision = self.engine.check(action, args, context)
        self.decisions.append(decision)
        return decision.allowed

print("PolicyAdapter defined — bridges PolicyDecision to bool for ToolGateway")

PolicyAdapter defined — bridges PolicyDecision to bool for ToolGateway


In [ ]:
from importlib.resources import files
# Lab 13.2a — Gateway + Policy: block submit_escalation
from knowlytix.harness.doe.gateway import ToolGateway, FaultProfile
from knowlytix.harness.fixtures import FullMockHarness

# Policy: require human_approval for submit_escalation
policy_rules = [
    PolicyRule(
        rule_id="GW-001", action="submit_escalation",
        requirement="human_approval", rule_type="requires",
        severity="error",
    ),
]
gw_engine = make_policy_engine(store, policy_rules)
adapter = PolicyAdapter(gw_engine)

gateway = ToolGateway(
    store=store,
    fault_profiles=[],
    policy_engine=adapter,
    run_id="lab13-2a",
)

# Use escalate mode so the harness attempts submit_escalation
harness = FullMockHarness(recovery_mode="escalate")
gateway.install(harness)

context = open(str(files("knowlytix.harness.doe").joinpath("tests/model_risk_assessment.md"))).read()
prompt = (
    f"<report>\n{context}\n</report>\n\n"
    "Answer the following question using the report and any tools "
    "available to you.\n\n"
    "What is the risk rating of MDL-001?"
)
answer = harness.run(prompt)
print(f"Answer: {answer[:100]!r}")

# Gateway transcript
tr = gateway.get_transcript()
print(f"\n=== Gateway Transcript ({len(tr.decisions)} decisions) ===")
for i, d in enumerate(tr.decisions):
    print(f"  [{d.action:12s}] {d.tool_name:<25s} {d.reason[:60]}")

# Policy decisions collected by the adapter
print(f"\n=== Policy Decisions ({len(adapter.decisions)}) ===")
for pd in adapter.decisions:
    print(f"  {pd.action:<25s} allowed={pd.allowed}  "
          f"blocking={pd.blocking_rules}")

gateway.uninstall(harness)

In [17]:
# Lab 13.2b — Gateway + Policy + Faults combined
# Policy blocks submit_escalation; faults inject errors on risk_check
combined_rules = [
    PolicyRule(
        rule_id="CMB-001", action="submit_escalation",
        requirement="human_approval", rule_type="requires",
        severity="error",
    ),
]

fault_profiles = [
    FaultProfile(
        tool_pattern="risk_check",
        error_rate=1.0,
        error_message="Risk service offline",
    ),
]

# Test all three recovery modes
print(f"{'Mode':<12s} {'Answer':<55s} {'GW Decisions':>12s}")
print("-" * 82)

for mode in ["retry", "fallback", "escalate"]:
    gw = ToolGateway(
        store=store,
        fault_profiles=fault_profiles,
        policy_engine=PolicyAdapter(make_policy_engine(store, combined_rules)),
        run_id=f"lab13-2b-{mode}",
    )
    h = FullMockHarness(recovery_mode=mode)
    gw.install(h)

    ans = h.run(prompt)
    tr = gw.get_transcript()

    actions = Counter(d.action for d in tr.decisions)
    action_str = ", ".join(f"{k}={v}" for k, v in sorted(actions.items()))
    print(f"{mode:<12s} {ans[:53]!r:<55s} {action_str}")

    gw.uninstall(h)

Mode         Answer                                                  GW Decisions
----------------------------------------------------------------------------------
retry        "{'name': 'CreditScore-XGB', 'domain': 'Credit Decisio" allow=1, inject_fault=1
fallback     "Tool 'risk_check' blocked; falling back to memory. # " allow=2, inject_fault=1
escalate     "{'name': 'CreditScore-XGB', 'domain': 'Credit Decisio" allow=1, block=1, inject_fault=1


In [18]:
# Lab 13.2c — Detailed transcript for escalate mode
# (escalate tries submit_escalation which is policy-blocked)
esc_rules = [
    PolicyRule(
        rule_id="ESC-001", action="submit_escalation",
        requirement="human_approval", rule_type="requires",
        severity="error",
    ),
]
esc_adapter = PolicyAdapter(make_policy_engine(store, esc_rules))

esc_faults = [
    FaultProfile(tool_pattern="risk_check", error_rate=1.0,
                 error_message="Risk service offline"),
]

gw_esc = ToolGateway(
    store=store, fault_profiles=esc_faults,
    policy_engine=esc_adapter, run_id="lab13-2c-escalate",
)
h_esc = FullMockHarness(recovery_mode="escalate")
gw_esc.install(h_esc)

ans_esc = h_esc.run(prompt)
tr_esc = gw_esc.get_transcript()

print(f"=== Escalate + Policy Block ({len(tr_esc.decisions)} gateway decisions) ===")
for d in tr_esc.decisions:
    print(f"  [{d.action:12s}] {d.tool_name:<25s} {d.reason[:60]}")

print(f"\n=== Policy Decisions ({len(esc_adapter.decisions)}) ===")
for pd in esc_adapter.decisions:
    rc = ", ".join(pd.rules_checked) if pd.rules_checked else "(none)"
    br = ", ".join(pd.blocking_rules) if pd.blocking_rules else "(none)"
    print(f"  {pd.action:<25s} allowed={pd.allowed}  checked=[{rc}]  blocked=[{br}]")

print(f"\nAnswer: {ans_esc[:100]!r}")

gw_esc.uninstall(h_esc)

=== Escalate + Policy Block (3 gateway decisions) ===
  [allow       ] data_lookup               Passed all gateway checks
  [inject_fault] risk_check                Risk service offline
  [block       ] submit_escalation         Blocked by policy engine

=== Policy Decisions (3) ===
  data_lookup               allowed=True  checked=[(none)]  blocked=[(none)]
  risk_check                allowed=True  checked=[(none)]  blocked=[(none)]
  submit_escalation         allowed=False  checked=[ESC-001]  blocked=[ESC-001]

Answer: "{'name': 'CreditScore-XGB', 'domain': 'Credit Decisioning', 'risk': 'High', 'status': 'Degraded', 'a"


---

## End-to-End: 640 Questions with Policy Enforcement

We run the full DOE benchmark through the adapter with policy enforcement and fault injection combined. The policy blocks `submit_escalation` while the fault profile injects 30% errors on `risk_check` — forcing the agent into recovery paths that may trigger the policy-blocked escalation.

In [ ]:
# E2E: 640 questions with policy + fault injection
import random
random.seed(42)

from knowlytix.harness.doe.adapters import AgentAdapter

e2e_rules = [
    PolicyRule(
        rule_id="E2E-001", action="submit_escalation",
        requirement="human_approval", rule_type="requires",
        severity="error",
    ),
]
e2e_adapter_policy = PolicyAdapter(make_policy_engine(store, e2e_rules))

e2e_faults = [
    FaultProfile(tool_pattern="risk_check", error_rate=0.3,
                 error_message="Risk service degraded"),
]

e2e_gateway = ToolGateway(
    store=store, fault_profiles=e2e_faults,
    policy_engine=e2e_adapter_policy,
    run_id="lab13-e2e",
)

e2e_agent_adapter = AgentAdapter(
    harness=FullMockHarness(recovery_mode="escalate"),
    gateway=e2e_gateway,
)

doe_questions = bench.generate_questions()
traces = []
for dq in doe_questions:
    t = e2e_agent_adapter.evaluate(
        question=dq.question.natural_language,
        context=context,
    )
    traces.append(t)

print(f"Collected {len(traces)} traces")
print(f"Errors   : {sum(1 for t in traces if t.error)}")
print(f"Non-empty: {sum(1 for t in traces if t.raw_output)}")

In [20]:
# Analyze the gateway transcript
tr_e2e = e2e_gateway.get_transcript()
print(f"=== Gateway Transcript ({len(tr_e2e.decisions)} decisions) ===")

action_counts = Counter(d.action for d in tr_e2e.decisions)
for action, count in action_counts.most_common():
    print(f"  {action:15s} {count:>5d}  ({100*count/len(tr_e2e.decisions):.1f}%)")

# By tool
print("\nBy tool:")
tool_action = {}
for d in tr_e2e.decisions:
    key = (d.tool_name, d.action)
    tool_action[key] = tool_action.get(key, 0) + 1

for (tool, action), count in sorted(tool_action.items()):
    print(f"  {tool:22s} {action:15s} {count:>5d}")

=== Gateway Transcript (2132 decisions) ===
  allow            1716  (80.5%)
  inject_fault      208  (9.8%)
  block             208  (9.8%)

By tool:
  approve_action         allow               4
  data_lookup            allow             640
  risk_check             allow             432
  risk_check             inject_fault      208
  send_notification      allow             640
  submit_escalation      block             208


In [21]:
# Analyze the policy decisions
policy_decisions = e2e_adapter_policy.decisions
print(f"=== Policy Decision Summary ({len(policy_decisions)} checks) ===")

allowed_count = sum(1 for d in policy_decisions if d.allowed)
blocked_count = sum(1 for d in policy_decisions if not d.allowed)
print(f"  Allowed : {allowed_count}")
print(f"  Blocked : {blocked_count}")

# Break down by action
print("\nBy action:")
action_allowed = {}
action_blocked = {}
for d in policy_decisions:
    if d.allowed:
        action_allowed[d.action] = action_allowed.get(d.action, 0) + 1
    else:
        action_blocked[d.action] = action_blocked.get(d.action, 0) + 1

all_actions = sorted(set(list(action_allowed.keys()) + list(action_blocked.keys())))
print(f"  {'Action':<25s} {'Allowed':>8s} {'Blocked':>8s}")
print(f"  {'-'*43}")
for a in all_actions:
    al = action_allowed.get(a, 0)
    bl = action_blocked.get(a, 0)
    print(f"  {a:<25s} {al:>8d} {bl:>8d}")

=== Policy Decision Summary (2132 checks) ===
  Allowed : 1924
  Blocked : 208

By action:
  Action                     Allowed  Blocked
  -------------------------------------------
  approve_action                   4        0
  data_lookup                    640        0
  risk_check                     640        0
  send_notification              640        0
  submit_escalation                0      208


In [22]:
# Tool call distribution in traces
all_tools = Counter(c.tool_name for t in traces for c in t.tool_calls)
print("=== Tool Calls in Traces ===")
for tool, count in all_tools.most_common():
    print(f"  {tool:22s} {count:>5d}")

# Latency statistics
latencies = [t.latency_ms for t in traces]
print("\n=== Latency (ms) ===")
print(f"  Mean   : {np.mean(latencies):.2f}")
print(f"  Std    : {np.std(latencies):.2f}")
print(f"  Median : {np.median(latencies):.2f}")
print(f"  P99    : {np.percentile(latencies, 99):.2f}")
print(f"  Max    : {np.max(latencies):.2f}")

=== Tool Calls in Traces ===
  data_lookup              640
  send_notification        640
  risk_check               432
  approve_action             4

=== Latency (ms) ===
  Mean   : 0.83
  Std    : 0.07
  Median : 0.79
  P99    : 1.03
  Max    : 1.29


---

## Exercises

**Exercise 13.1** — Create a policy with five rules: two `requires`, two `blocks` (one scoped to `context="production"`) and one `allows`. Run `check()` for each tool under both `production` and `staging` contexts. Verify that context-scoped rules only trigger in their intended context.

**Exercise 13.2** — Write a YAML policy file and load it with `load_policy_rules()`. Create a `PolicyEngine` and verify that `encode_rules()` creates the expected GMS triples by querying `store.query_triples()` for each rule's action entity.

**Exercise 13.3** — Run the 640-question benchmark with policy blocking `risk_check` (the most commonly called tool) under all three recovery modes. Compare the gateway transcripts: how many policy blocks occur, and how does each recovery mode handle the blocked tool?

---

## Summary

This chapter introduced the **PolicyEngine** — the governance layer that validates every tool call before execution:

1. **PolicyRule** defines a single constraint: an action pattern, a requirement, a rule type (`requires`/`blocks`/`allows`) and a severity level. Rules are matched by `fnmatch` glob and can be scoped to specific contexts.

2. **PolicyEngine** checks each action through four layers: explicit rule matching, GMS graph lookup, geometric plausibility scoring (threshold 1.5) and contradiction detection via tension energy (threshold 1.7). Rules are encoded as GMS triples so the same geometric primitives used for knowledge verification also enforce policy.

3. **PolicyDecision** captures the full audit trail: pass/fail, blocking rule IDs, geometric scores and a human-readable reason.

4. **Gateway integration** — The `ToolGateway` delegates to the policy engine as its first check in the `before_tool` hook. Policy violations are blocked before any fault injection or mock substitution takes place.

In the next chapter we add **EnvironmentSimulator** — deterministic world-state with tool endpoint specs, state transitions and reproducible test data.